#Inference Engineering

Task 1: Implement quantization from scratch

In [5]:
import numpy as np

def quantize_to_int8(data):
    data = np.array(data, dtype=np.float32) #Converts input to a numpy float32 array.
    qmin, qmax = -128, 127

    x_min = float(data.min())
    x_max = float(data.max()) #Finds the actual spread of your data.
    x_min = min(x_min, 0.0)
    x_max = max(x_max, 0.0) #range to include 0.0.

    scale = (x_max - x_min) / (qmax - qmin)
    if scale == 0:
        scale = 1e-8  #can not divide by zero

    zero_point = round(qmin - x_min / scale)
    zero_point = int(np.clip(zero_point, qmin, qmax))

    q = np.round(data / scale + zero_point)
    q = np.clip(q, qmin, qmax).astype(np.int8)
    return q, scale, zero_point


def dequantize_int8(q, scale, zero_point): #Dequantization
    q = np.asarray(q, dtype=np.int32)
    return (q - zero_point) * scale

#Array
test_data = [-2.5, -1.0, 0.0, 0.3, 1.7, 3.4, 5.0]
q, scale, zp = quantize_to_int8(test_data)
deq = dequantize_int8(q, scale, zp)

print("Original: ", test_data)
print("Scale:    ", scale)
print("ZeroPoint:", zp)
print("Quantized:", q)
print("Dequant:  ", np.round(deq, 4))
print("Abs error:", np.round(np.abs(np.array(test_data) - deq), 4))

Original:  [-2.5, -1.0, 0.0, 0.3, 1.7, 3.4, 5.0]
Scale:     0.029411764705882353
ZeroPoint: -43
Quantized: [-128  -77  -43  -33   15   73  127]
Dequant:   [-2.5    -1.      0.      0.2941  1.7059  3.4118  5.    ]
Abs error: [0.     0.     0.     0.0059 0.0059 0.0118 0.    ]


Scale : How much the float (original) value change for one step in int
Zero_point : which int8 integer corresponds to the float value 0.0

The float32 array was quantized into int8 by computing a scale factor and a zero-point from the data's min/max range

Task 2: Estimate model size from parameter count

In [14]:
BYTES_PER_PARAM = {
    "fp32": 4,
    "fp16": 2,
    "int8": 1,
}

def model_size(num_params, dtype="fp32"):
    """
    Return (size_in_MB, size_in_GB) for a model with `num_params` parameters
    stored in the given dtype.
    """
    if dtype not in BYTES_PER_PARAM:
        raise ValueError(f"Unsupported dtype: {dtype}. Choose from {list(BYTES_PER_PARAM)}")

    total_bytes = num_params * BYTES_PER_PARAM[dtype]
    size_mb = total_bytes / (1024 ** 2)
    size_gb = total_bytes / (1024 ** 3)
    return size_mb, size_gb


example_models = {
"125M": 125_000_000,
"1B":   1_000_000_000,
"7B":   7_000_000_000,
}

print(f"{'Model':<8}{'dtype':<8}{'Size (MB)':>14}{'Size (GB)':>14}")
print("-" * 44)
for name, params in example_models.items():
    for dtype in ["fp32", "fp16", "int8"]:
        mb, gb = model_size(params, dtype)
        print(f"{name:<8}{dtype:<8}{mb:>14,.1f}{gb:>14,.2f}")
    print()

Model   dtype        Size (MB)     Size (GB)
--------------------------------------------
125M    fp32             476.8          0.47
125M    fp16             238.4          0.23
125M    int8             119.2          0.12

1B      fp32           3,814.7          3.73
1B      fp16           1,907.3          1.86
1B      int8             953.7          0.93

7B      fp32          26,702.9         26.08
7B      fp16          13,351.4         13.04
7B      int8           6,675.7          6.52



when we convert FP32 to INT 8 the precision decrease, size of model drops significantly.FP32 requires 4 bytes per parameter, while INT8 requires only 1 byte per parameter, resulting in an approximately 75% reduction in weight storage.

#Task 3: System design scenario — multi-GPU inference

Scenario: You need to serve a large language model that doesn't fit on a single GPU. You have access to a node with 8 GPUs connected via NVLink, and multiple such nodes connected via InfiniBand.

##1. Where would you split the model, and why?

Design Note

If a large language model that cannot fit on a single GPU, I would first split the model within the same node across the 8 GPUs connected by NVLink. NVLink provides high-bandwidth, low-latency GPU-to-GPU communication, so it is suitable for communication-heavy techniques such as tensor parallelism, where different GPUs work on parts of the same layer.

If the model is still too large for one 8-GPU node, I would then extend the parallelism across nodes using InfiniBand. Techniques such as pipeline parallelism can be used across nodes, while trying to minimize the amount of communication that crosses the node boundary.

##2.Role of NVLink vs InfiniBand

NVLink is used for fast communication between GPUs within the same node.

InfiniBand is used for high-speed communication between different nodes or servers.

##3.What would go wrong if you swapped their roles?

If we tried to use InfiniBand for communication insted of NVLink, the lower bandwidth and higher latency would increase communication overhead and could significantly reduce inference performance.

Similarly, Using NVLink as the inter-node network is not possible because NVLink connects GPUs within the appropriate node-level, whereas InfiniBand provides the network connection between separate servers.

In short: keep communication-heavy operations inside the node over NVLink, and use InfiniBand for communication that must cross nodes. This minimizes latency and maximizes multi-GPU inference performance.

Task 4: Compute-bound vs memory-bound estimation (30 min)

In [18]:
def check(flops, gpu, memory, bandwidth):
    time_compute = flops / gpu
    time_memory = memory / bandwidth

    if time_compute > time_memory:
        result = "Compute-bound"
    elif time_memory > time_compute:
        result = "Memory-bound"
    else:
        result = "Balanced"

    return result, time_compute, time_memory


# Configuration 1
result1 = check(200e9, 20e12, 10e9, 1e12)

print("Configuration 1")
print("Inference type:", result1[0])
print("Compute time:", round(result1[1] * 1000, 2), "ms")
print("Memory time :", round(result1[2] * 1000, 2), "ms")
print()


# Configuration 2
result2 = check(500e9, 20e12, 10e9, 1e12)

print("Configuration 2")
print("Inference type:", result2[0])
print("Compute time:", round(result2[1] * 1000, 2), "ms")
print("Memory time :", round(result2[2] * 1000, 2), "ms")
print()


# Configuration 3
result3 = check(100e9, 20e12, 20e9, 1e12)

print("Configuration 3")
print("Inference type:", result3[0])
print("Compute time:", round(result3[1] * 1000, 2), "ms")
print("Memory time :", round(result3[2] * 1000, 2), "ms")

Configuration 1
Inference type: Balanced
Compute time: 10.0 ms
Memory time : 10.0 ms

Configuration 2
Inference type: Compute-bound
Compute time: 25.0 ms
Memory time : 10.0 ms

Configuration 3
Inference type: Memory-bound
Compute time: 5.0 ms
Memory time : 20.0 ms


Small batch :memory access was the slow part (7 ms vs 0.045 ms compute), so it was memory-bound.

With a large batch: compute became the slow part (11.5 ms vs 7 ms), so it was compute-bound.